In [ ]:
import time
from pathlib import Path
from urllib.request import urlretrieve

from src.runtime.runtime_config import load_config
from src.preprocessing.DataLoader import DataLoader
from src.reconstruction.JointReconstructor import JointReconstructor
from src.utils.notebook_display import display_input_sampling_motion_panels, display_run_panels
from src.runtime.runtime_setup import initialize_runtime

jupyter_notebook_flag = True
DEBUG_FLAG = False

# ISMRMRD_FILE = Path("../data/Breast-INNOV_GRICS_database/ISMRMRD/0073_T2_s.h5")
# SAEC_FILE = Path("../data/Breast-INNOV_GRICS_database/SAEC/0073_T2_s.h5")

ISMRMRD_FILE = Path("../data/GRICS-torch/test_XA61_volunteer/0274_T2_s.dat")
SAEC_FILE = Path("../data/GRICS-torch/test_XA61_volunteer/0274_T2_s.saec")

def main():

    print("[Demo E] Loading config...")
    params = load_config(
        data_type="siemens-saec",
        reconstruction_config="config/reconstruction/nonrigid_2d_breast.toml",
        overrides={
            "jupyter_notebook_flag": jupyter_notebook_flag,
            "debug_flag": DEBUG_FLAG,
        },
    )

    print("[Demo E] Initializing runtime...")
    sp_device, t_device = initialize_runtime(params)

    print("[Demo E] Loading data and building operators...")
    data = DataLoader(
        params=params,
        t_device=t_device,
        sp_device=sp_device,
        filename=(str(ISMRMRD_FILE), str(SAEC_FILE)),
        slice_idx=15
    )
    display_input_sampling_motion_panels(
        params,
    )

    print("[Demo E] Starting reconstruction...")
    recon = JointReconstructor(
        data.kspace,
        data.smaps,
        data.sampling_idx,
        motion_signal=data.motion_signal,
        params=params,
        kspace_scale=data.kspace_scale,
        motion_plot_context=data.motion_plot_context,
    )
    t0 = time.time()
    recon.run()
    print(f"Elapsed time: {time.time() - t0:.2f} s")
    display_run_panels(
        params,
        motion_type=params.reconstruction_motion_type,
    )


if __name__ == "__main__":
    main()

